# Step 2: Select Dark Matter Subhalos

This notebook fetches the TNG-100 subhalo catalog and applies selection criteria to identify suitable dark matter subhalos for the XAI dataset.

## Process

1. **Fetch Initial Catalog**: Download subhalos from TNG API ordered by stellar mass
2. **Apply Filters**: Select subhalos based on dark matter properties
3. **Save Selection**: Export filtered catalog for next steps

In [ ]:
# SETUP CELDA 1: Configure Google Colab environment
import os
import sys
from pathlib import Path

# Detect if running in Google Colab
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("🔧 Setting up Google Colab environment...\n")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✓ Google Drive mounted at /content/drive")
    
    # Change to project directory
    project_path = '/content/drive/MyDrive/xai-dark-matter-localization'
    os.chdir(project_path)
    print(f"✓ Working directory: {os.getcwd()}")
    
    # Add to Python path
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print("✓ Python path configured\n")
else:
    print("⚠️  Not running in Colab. Using local environment.\n")

In [ ]:
# SETUP CELDA 2: Import required libraries and project modules
import pandas as pd
import numpy as np
from tqdm import tqdm

# Import project modules
from src.config import DATA_ROOT, DATASET_DIR, TNG_API_KEY
from src.select_subhalos import build_initial_catalog

print("✓ All libraries and modules imported successfully")
print(f"\nProject Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  API key configured: {'✓' if TNG_API_KEY else '✗'}")

## Celda 2: Import Project Modules

Now import all required libraries and project modules.

In [ ]:
# Save selected catalog
output_file = raw_metadata_dir / "subhalos_selected.csv"
df_selected.to_csv(output_file, index=False)
print(f"✓ Saved {len(df_selected)} selected subhalos to:")
print(f"  {output_file}")

# Also save the full catalog for reference
full_catalog_file = raw_metadata_dir / "subhalos_full_catalog.csv"
df_catalog.to_csv(full_catalog_file, index=False)
print(f"\n✓ Saved full catalog ({len(df_catalog)} subhalos) to:")
print(f"  {full_catalog_file}")

# Create summary report
print("\n" + "="*50)
print("SELECTION SUMMARY")
print("="*50)
print(f"Full catalog size: {len(df_catalog)}")
print(f"Selected subhalos: {len(df_selected)}")
print(f"Selection ratio: {100 * len(df_selected) / len(df_catalog):.1f}%")
print(f"\nNext step: Download images for {len(df_selected)} subhalos")
print(f"See notebook: 03_download_images.ipynb")

## 5. Save Selection

Export the selected subhalos to CSV for use in the next pipeline steps.

In [ ]:
# Show sample of selected subhalos
print("\nSample of selected subhalos:")
print(df_selected[['subhalo_id', 'mass_log_msun', 'stellar_mass_api', 'sfr', 'group_nsubs']].head(10))

In [ ]:
# Display selection statistics
print("Selected Subhalos Statistics:")
print("="*50)
print(f"\nMass distribution (log10 M_sun):")
print(f"  Mean: {df_selected['mass_log_msun'].mean():.2f}")
print(f"  Median: {df_selected['mass_log_msun'].median():.2f}")
print(f"  Std: {df_selected['mass_log_msun'].std():.2f}")
print(f"  Range: [{df_selected['mass_log_msun'].min():.2f}, {df_selected['mass_log_msun'].max():.2f}]")

print(f"\nStellar Mass distribution (log10 M_sun):")
print(f"  Mean: {df_selected['stellar_mass_api'].mean():.2e}")
print(f"  Median: {df_selected['stellar_mass_api'].median():.2e}")

print(f"\nStar Formation Rate (SFR):")
print(f"  Mean: {df_selected['sfr'].mean():.2f} M_sun/yr")
print(f"  Median: {df_selected['sfr'].median():.2f} M_sun/yr")
print(f"  Max: {df_selected['sfr'].max():.2f} M_sun/yr")

print(f"\nParent Halo Properties:")
print(f"  Avg subhalos per halo: {df_selected['group_nsubs'].mean():.1f}")
print(f"  Avg halo mass (M_crit200): {df_selected['group_m_crit200'].mean():.2e} M_sun")

print(f"\nPosition distribution (kpc/h):")
pos_distance = np.sqrt(
    df_selected['pos_x']**2 + 
    df_selected['pos_y']**2 + 
    df_selected['pos_z']**2
)
print(f"  Mean distance: {pos_distance.mean():.1f} kpc/h")
print(f"  Max distance: {pos_distance.max():.1f} kpc/h")

## 4. Analyze Selected Sample

Examine the properties of the selected dark matter subhalos.

In [ ]:
# Define selection criteria for dark matter subhalos
print("Applying selection filters...\n")

# Start with full catalog
df_selected = df_catalog.copy()
print(f"Initial catalog: {len(df_selected)} subhalos")

# Filter 1: Remove central galaxies (we want subhalos, not centrals)
df_selected = df_selected[df_selected['is_central'] == 0]
print(f"After removing centrals: {len(df_selected)} subhalos")

# Filter 2: Require valid position data
df_selected = df_selected[
    (df_selected['pos_x'].notna()) & 
    (df_selected['pos_y'].notna()) & 
    (df_selected['pos_z'].notna())
]
print(f"After requiring valid positions: {len(df_selected)} subhalos")

# Filter 3: Require dark matter mass estimate (mass_log_msun)
# This indicates the total mass including dark matter
df_selected = df_selected[df_selected['mass_log_msun'].notna()]
print(f"After requiring mass estimates: {len(df_selected)} subhalos")

# Filter 4: Select mass range for suitable dark matter halos
# log10(M_sun) between 11 and 13 = 10^11 to 10^13 solar masses
MIN_LOG_MASS = 11.0
MAX_LOG_MASS = 13.5
df_selected = df_selected[
    (df_selected['mass_log_msun'] >= MIN_LOG_MASS) & 
    (df_selected['mass_log_msun'] <= MAX_LOG_MASS)
]
print(f"After mass range filter ({MIN_LOG_MASS} - {MAX_LOG_MASS}): {len(df_selected)} subhalos")

# Filter 5: Require parent halo information (quality check)
df_selected = df_selected[
    (df_selected['group_m_crit200'].notna()) & 
    (df_selected['group_nsubs'].notna())
]
print(f"After requiring parent halo info: {len(df_selected)} subhalos")

# Filter 6: Ensure quality flag is set
df_selected = df_selected[df_selected['quality_flag'] == 1]
print(f"After quality flag: {len(df_selected)} subhalos")

print(f"\n✓ Selection complete: {len(df_selected)} subhalos selected")
print(f"  Percentage retained: {100 * len(df_selected) / len(df_catalog):.1f}%")

## 3. Apply Selection Filters

Select subhalos based on criteria for dark matter localization:
- **Type**: Subhalos vs central galaxies
- **Mass**: Reasonable dark matter halo mass range
- **Data Quality**: Valid positions and mass estimates
- **Halo Environment**: Within larger halo structures

In [ ]:
# 3. Explore catalog structure
print("Catalog columns:")
print(df_catalog.columns.tolist())
print("\nData types:")
print(df_catalog.dtypes)
print("\nCatalog statistics:")
print(df_catalog.describe())

In [ ]:
# 2. Fetch subhalo catalog
# Set parameters for selection
MAX_SUBHALOS = 1000  # Number of initial subhalos to fetch (adjust for speed)
PAGE_SIZE = 100      # How many to fetch per API call

print(f"Fetching {MAX_SUBHALOS} subhalos from TNG-100 API...")
print(f"(This may take 5-15 minutes depending on API rate limits)")
print()

# Ensure output directory exists
raw_metadata_dir = DATA_ROOT / "raw" / "tng" / "metadata_raw"
raw_metadata_dir.mkdir(parents=True, exist_ok=True)

# Fetch catalog
df_catalog = build_initial_catalog(max_subhalos=MAX_SUBHALOS, page_size=PAGE_SIZE)

print(f"\n✓ Fetched {len(df_catalog)} subhalos")
print(f"\nFirst few rows:")
print(df_catalog.head())

## 2. Fetch Initial Subhalo Catalog

Download the TNG-100 subhalo catalog from the API. This may take several minutes as it fetches detailed information for each subhalo.